# Arpa-150M — pré-treino no Colab

Dois estágios, pra **não desperdiçar A100**:

- **FASE A — runtime CPU (grátis):** tokenizer + bins. Não usa GPU (é download + tokenização). Salva tudo no teu Drive.
- **FASE B — runtime A100:** só o treino, que é o que precisa de GPU.

Ambas as fases são **idempotentes e resumíveis**: se o Colab cair, troque o runtime de volta e rode de novo — a geração dos bins continua de onde parou (sidecar `progress.json`) e o treino retoma do último checkpoint.

Tudo persiste em `MyDrive/arpa150m/`.

## Setup comum (rodar nas duas fases)

In [ ]:
!pip install -q torch transformers datasets tokenizers

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/arpa150m'
TOK_DRIVE  = f'{DRIVE_ROOT}/tokenizer-arpa-64k-clean'
DATA_DRIVE = f'{DRIVE_ROOT}/data'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Persistência em:', DRIVE_ROOT)

In [ ]:
REPO = '/content/ResNet-Psi'
if os.path.exists(REPO):
    !cd {REPO} && git pull --ff-only
else:
    !git clone https://github.com/Lucas-404/ResNet-Psi.git {REPO}
%cd {REPO}
!git log -1 --oneline

In [ ]:
# (Opcional, mas RECOMENDADO) token do HuggingFace — evita rate-limit no download
# e acelera a Fase A. Pegue em https://huggingface.co/settings/tokens (read).
# os.environ['HF_TOKEN'] = 'hf_xxxxxxxxxxxxxxxx'
print('HF_TOKEN:', 'definido' if os.environ.get('HF_TOKEN') else 'NAO definido (vai usar acesso anonimo)')

# FASE A — tokenizer + bins (runtime CPU grátis)

Troque para um runtime **sem GPU** (Ambiente de execução → Alterar tipo → CPU) e rode as duas células abaixo. Pode demorar horas (download dos datasets); se cair, rode de novo que retoma.

In [ ]:
# Tokenizer 64K — gera 1x direto no Drive
if os.path.exists(os.path.join(TOK_DRIVE, 'tokenizer.json')):
    print('Tokenizer já existe no Drive — pulando')
else:
    print('Treinando tokenizer 64K (uma vez)...')
    !python arpa/train_tokenizer.py --output-dir "{TOK_DRIVE}"
assert os.path.exists(os.path.join(TOK_DRIVE, 'tokenizer.json'))
print('tokenizer OK')

In [ ]:
# Bins de pré-treino — escritos DIRETO no Drive, com auto-resume.
# O script se cura sozinho de quedas de rede da HF; se a SESSÃO do Colab cair,
# rode esta MESMA célula de novo: continua de onde parou.
!python arpa/prepare_data.py --train-tokens 3.3e9 \
    --out-dir "{DATA_DRIVE}" --tokenizer-dir "{TOK_DRIVE}"
done = os.path.exists(os.path.join(DATA_DRIVE, 'DONE'))
print('BINS COMPLETOS!' if done else 'INCOMPLETO — rode esta célula de novo (retoma sozinho)')

# FASE B — pré-treino (runtime A100)

Quando a Fase A terminar, troque o runtime para **GPU A100** (Ambiente de execução → Alterar tipo → GPU → A100), rode o *Setup comum* de novo e siga daqui.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), 'Sem GPU — troque o runtime para A100'
print('GPU OK | BF16:', torch.cuda.is_bf16_supported())

In [ ]:
# Restaura tokenizer + bins do Drive para o disco local (leitura rápida no treino)
import shutil
TOK = 'tokenizer-arpa-64k-clean'
if not os.path.exists(os.path.join(TOK, 'tokenizer.json')):
    shutil.copytree(TOK_DRIVE, TOK)
os.makedirs('data/arpa150m', exist_ok=True)
for f in ('train_tokens.bin', 'val_tokens.bin'):
    dst = f'data/arpa150m/{f}'
    if not os.path.exists(dst):
        print('copiando', f, '...')
        shutil.copy(f'{DATA_DRIVE}/{f}', dst)
print('tokenizer + bins prontos localmente')

In [ ]:
# Checkpoints vão direto pro Drive (sobrevivem a desconexão)
ckpt_drive = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(ckpt_drive, exist_ok=True)
link = 'checkpoints-arpa150m'
if os.path.exists(link) and not os.path.islink(link):
    shutil.rmtree(link)
if not os.path.exists(link):
    os.symlink(ckpt_drive, link)
print(link, '->', os.path.realpath(link))

In [ ]:
# Pré-treino A100 (~12-18h). Auto-resume: roda quantas vezes precisar.
import glob
resume = '--resume latest' if glob.glob('checkpoints-arpa150m/step_*.pt') else ''
print('RETOMANDO treino' if resume else 'COMEÇANDO do zero')
!python arpa/pretrain.py --config a100 {resume}

## (Depois do pré-treino) SFT + chat

In [ ]:
!python arpa/sft.py --init checkpoints-arpa150m/best.pt --data "sft/*.jsonl"

In [ ]:
!python arpa/sample.py --ckpt checkpoints-sft/best.pt --chat